# guide

> The screen guide: "I don't get X", answered with a board that the tutor draws over the page.

The learner looks at a walk step, a concept or a video, and says "I don't get alpha bar". The guide answers on the page,
as a teacher does at a whiteboard. It circles the term, draws an arrow to a label in the figure, moves the slider of the
demo, and speaks one short line for each step.

This module is the part of the guide that has no web page:

- the **board language**: a closed list of drawing commands, each tied to a thing on the page (an *anchor*), and the
  check of each line of the model's reply;
- the **context**: what the page shows, cleaned and cut to size;
- the **gate**: the tutor answers at once when it is sure, else it looks in your vault, else on the web;
- the **guard**: no secret and no personal data leaves the Mac in a model call or a web query.

The `guide` block serves it, and `core/board.js` draws it.

In [ ]:
#| default_exp guide

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # nor the real ramabana config

In [ ]:
#| hide
from fastcore.test import *

Import the helpers. The model layer and ramabana load only on the first call, so the import stays light.

In [ ]:
#| export
import hashlib, json, math, re, threading, time
from pathlib import Path
from fastcore.utils import *

## The board language

A reply is JSON lines: one object on each line. The host checks each line alone, so a bad line drops and the rest
draws. There are 11 commands. The model never writes HTML, CSS, script or a URL.

| op | fields | what it does |
|---|---|---|
| `step` | `say`, and `src` with `quote` | starts a step; `say` is the spoken line. A quote is word for word from source `src` |
| `circle`, `box`, `underline` | `at` | a hand-drawn ring, box or line on the anchor |
| `arrow` | `from`, `to` | a curved arrow between two anchors |
| `label` | `at`, `text`, `side` | a short note beside the anchor |
| `sketch` | `at`, `pts` | a free line in the box of the anchor, for a small curve |
| `point` | `at` | moves a pointer mark to the anchor |
| `clear` | `n` (optional) | removes the marks of all steps, or of step `n` |
| `demo_set` | `set` | changes inputs of the step's rupaka demo, only after its gate is open |
| `video_at` | `t`, `quote` | moves the explainer video to `t` seconds, and quotes its transcript there |

Each stroke can have `w`: the word of `say` where it starts.

In [ ]:
#| export
OPS = ('step', 'circle', 'box', 'underline', 'arrow', 'label', 'sketch', 'point', 'clear', 'demo_set', 'video_at')
FIELDS = dict(step={'say', 'src', 'quote'}, circle={'at'}, box={'at'}, underline={'at'}, arrow={'from', 'to'},
              label={'at', 'text', 'side'}, sketch={'at', 'pts'}, point={'at'}, clear={'n'}, demo_set={'set'}, video_at={'t', 'quote'})
LIMITS = dict(steps=6, strokes=6, lines=60, line_bytes=600, pts=64, say=160, text=60, quote_min=40, quote_max=600, set=6)
SIDES = ('above', 'below', 'left', 'right')
SURE = 0.7   # the least confidence for an answer from the tutor alone

The first line can be a header: `{"op":"v","n":1,"sure":0.9,"need":"none"}`. `sure` is how sure the model is, from 0 to
1. `need` says what it needs to be sure: `none`, `vault` or `web`.

In [ ]:
#| export
NEEDS = ('none', 'vault', 'web')
VERSION = 1

## Anchors

A stroke names a thing on the page. The page sends a list of anchors, each with an `id`, a `kind` and its `text`. A
formula also has its `tokens` (the TeX of its parts), a figure has its `labels` (a box for each label in the image),
and the demo has its inputs.

| form | example | the place |
|---|---|---|
| an id | `"passage"` | the whole anchor |
| words | `{"a": "passage", "text": "closed form"}` | the words in the anchor |
| a maths token | `{"a": "m1", "tex": "\\bar\\alpha_t"}` | the part of the formula |
| a figure label | `{"a": "figure", "label": "a"}` | the label in the image |
| a box | `{"a": "figure", "box": [100, 200, 400, 600]}` | a part of the anchor, from 0 to 1000 |
| a point | `{"a": "m1", "pt": [1100, 500]}` | a point beside the anchor, for the far end of an arrow |

Two ways to write the same TeX must match. `norm_tex` takes out the spaces and the braces around one symbol.

In [ ]:
#| export
_ONE = re.compile(r'\{(\\[A-Za-z]+|[^{}\\])\}')

def norm_tex(s:str) -> str:
    "`s` with no white space, and with no braces around one character or one command, so `\\bar{\\alpha}_{t}` is `\\bar\\alpha_t`."
    s = re.sub(r'\s+', '', str(s or ''))
    while (t := _ONE.sub(r'\1', s)) != s: s = t
    return s

Braces and spaces do not change the match:

In [ ]:
test_eq(norm_tex(r'\bar{\alpha}_{t}'), norm_tex(r'\bar\alpha_t'))
test_eq(norm_tex(r'\sqrt{\bar\alpha_t}'), r'\sqrt{\bar\alpha_t}')   # a group of two symbols keeps its braces
test_ne(norm_tex(r'\alpha_t'), norm_tex(r'\bar\alpha_t'))

Words match with no case and with white space made single, as the page shows them.

In [ ]:
#| export
def flat(s) -> str: return ' '.join(str(s or '').split()).lower()

def _box(b, lo=0, hi=1000) -> bool:
    "Whether `b` is `[x0, y0, x1, y1]` with numbers from `lo` to `hi`, and `x0 < x1`, `y0 < y1`."
    return (isinstance(b, list) and len(b) == 4 and all(_num(x) and lo <= x <= hi for x in b) and b[0] < b[2] and b[1] < b[3])

def _num(x) -> bool: return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)

`anchor_why` gives the reason that an anchor is not on the page, or `''` when it is. The context `ctx` has the anchors
by id in `ctx.by`.

In [ ]:
#| export
FORMS = ('text', 'tex', 'label', 'box', 'pt')

def anchor_why(at, ctx) -> str:
    "Why `at` names no anchor of `ctx`, or '' when it does."
    if isinstance(at, str): return '' if at in ctx.by else f'no anchor {at!r}'
    if not isinstance(at, dict) or set(at) - {'a', *FORMS} or not isinstance(at.get('a'), str): return 'a bad anchor'
    if (a := ctx.by.get(at['a'])) is None: return f"no anchor {at['a']!r}"
    forms = [f for f in FORMS if f in at]
    if len(forms) > 1: return 'an anchor with two forms'
    if not forms: return ''
    f, v = forms[0], at[forms[0]]
    if f == 'text': return '' if isinstance(v, str) and v.strip() and flat(v) in flat(a.get('text')) else f'no words {v!r} in {at["a"]}'
    if f == 'tex': return '' if isinstance(v, str) and norm_tex(v) in {norm_tex(t) for t in a.get('tokens') or ()} else f'no token {v!r} in {at["a"]}'
    if f == 'label': return '' if isinstance(v, str) and v in (a.get('labels') or {}) else f'no label {v!r} in {at["a"]}'
    if f == 'box': return '' if _box(v) else 'a bad box'
    return '' if isinstance(v, list) and len(v) == 2 and all(_num(x) and -1000 <= x <= 2000 for x in v) else 'a bad point'

A small context for the tests: a passage, the DDPM formula with its tokens, a triangle figure with labels `a` and `b`,
and a demo with the input `t`.

In [ ]:
#| export
def by_id(ctx) -> AttrDict:
    "`ctx` with `by`: its anchors by id."
    ctx = AttrDict(ctx)
    ctx.by = {a['id']: a for a in ctx.get('anchors') or ()}
    return ctx

In [ ]:
DDPM = r'q(\mathbf{x}_t\mid\mathbf{x}_0)=\mathcal{N}(\mathbf{x}_t;\sqrt{\bar\alpha_t}\mathbf{x}_0,(1-\bar\alpha_t)\mathbf{I})'
ctx = by_id(dict(anchors=[dict(id='passage', kind='passage', text='It admits sampling x_t at an arbitrary timestep t in closed form.'),
                          dict(id='m1', kind='formula', text=DDPM, tokens=[r'\bar\alpha_t', r'\sqrt{\bar\alpha_t}', r'\mathbf{x}_0', r'(1-\bar\alpha_t)']),
                          dict(id='figure', kind='figure', text='A right triangle', labels=dict(a=[100, 420, 160, 500], b=[480, 880, 540, 960]))],
                 demo=dict(inputs=dict(t=dict(value=500, min=0, max=1000, label='Step t')), open=False)))

Each form finds its anchor, and a wrong name does not:

In [ ]:
test_eq(anchor_why('passage', ctx), '')
test_eq(anchor_why(dict(a='m1', tex=r'\bar{\alpha}_t'), ctx), '')
test_eq(anchor_why(dict(a='passage', text='Closed  form'), ctx), '')
test_eq(anchor_why(dict(a='figure', label='a'), ctx), '')
test_eq(anchor_why(dict(a='figure', box=[0, 0, 500, 500]), ctx), '')
assert anchor_why('e7', ctx)                                  # an id that the page did not send
assert anchor_why(dict(a='m1', tex=r'\beta_t'), ctx)          # a token that is not in the formula
assert anchor_why(dict(a='passage', text='open form'), ctx)   # words that are not in the passage
assert anchor_why(dict(a='figure', label='c'), ctx)           # a label that is not in the figure
assert anchor_why(dict(a='figure', box=[500, 0, 100, 10]), ctx)
assert anchor_why(dict(a='m1', tex=r'\bar\alpha_t', text='x'), ctx)   # two forms at once

## The check of one line

`check_line` gives the reason that a line drops, or `''`. Each string in a line is checked too: no tag, no script and no
link can get in, also as the text of a label.

In [ ]:
#| export
UNSAFE = re.compile(r'<[a-z/!?]|javascript:|https?://|www\.|data:', re.I)

def _strings(o):
    "Every string inside `o`, also the keys."
    if isinstance(o, str): yield o
    elif isinstance(o, dict):
        for k, v in o.items(): yield k; yield from _strings(v)
    elif isinstance(o, list):
        for v in o: yield from _strings(v)

A cue of the explainer video is `[t, text]`. `cue_at` gives the cue that plays at `t` seconds.

In [ ]:
#| export
def cue_at(cues, t:float):
    "The cue `(start, text)` that plays at `t`: the last one that starts at or before `t`. None when there is none."
    out = None
    for c in cues or ():
        if c[0] <= t: out = c
        else: break
    return out

def _in_cues(cues, t:float, quote:str) -> bool:
    "Whether `quote` (3 words or more) is word for word in the cue at `t` with the cue before and after it."
    if len(quote.split()) < 3 or not (c := cue_at(cues, t)): return False
    i = list(cues).index(c)
    near = ' '.join(x[1] for x in cues[max(0, i - 1):i + 2])
    return re.search(rf'(?<!\w){re.escape(flat(quote))}(?!\w)', flat(near)) is not None

In [ ]:
#| export
def _demo_why(o, ctx) -> str:
    d = ctx.get('demo') or {}
    ins, s = d.get('inputs') or {}, o.get('set')
    if not d.get('open'): return 'the demo gate is closed'
    if not isinstance(s, dict) or not 1 <= len(s) <= LIMITS['set']: return 'a bad set'
    for k, v in s.items():
        if k not in ins: return f'no demo input {k!r}'
        if not isinstance(v, (int, float, str, bool)) or (isinstance(v, str) and len(v) > 60): return f'a bad value for {k!r}'
        lo, hi = ins[k].get('min'), ins[k].get('max')
        if _num(v) and ((_num(lo) and v < lo) or (_num(hi) and v > hi)): return f'{k} out of range'
    return ''

def check_line(o, ctx) -> str:
    "Why the reply line `o` drops, or '' when it is good. `ctx` comes from `by_id`."
    if not isinstance(o, dict) or o.get('op') not in OPS: return 'an unknown op'
    op = o['op']
    if (bad := set(o) - {'op', 'w', 'n'} - FIELDS[op]): return f'unknown keys {sorted(bad)}'
    if any(UNSAFE.search(s) for s in _strings(o)): return 'a tag, a script or a link'
    if 'w' in o and not (isinstance(o['w'], int) and not isinstance(o['w'], bool) and 0 <= o['w'] < 100): return 'a bad word mark'
    if op == 'step':
        say = o.get('say')
        if not (isinstance(say, str) and say.strip() and len(say) <= LIMITS['say'] and say.isprintable()): return 'a bad say'
        return ''
    if op in ('circle', 'box', 'underline', 'point', 'label', 'sketch'):
        if (why := anchor_why(o.get('at'), ctx)): return why
    if op == 'arrow': return anchor_why(o.get('from'), ctx) or anchor_why(o.get('to'), ctx)
    if op == 'label':
        t = o.get('text')
        if not (isinstance(t, str) and t.strip() and len(t) <= LIMITS['text'] and t.isprintable()): return 'a bad label text'
        if o.get('side', 'above') not in SIDES: return 'a bad side'
    if op == 'sketch':
        p = o.get('pts')
        if not (isinstance(p, list) and 2 <= len(p) <= LIMITS['pts']): return 'a bad sketch'
        if not all(isinstance(q, list) and len(q) == 2 and all(_num(x) and -1000 <= x <= 2000 for x in q) for q in p): return 'a bad sketch point'
    if op == 'clear' and 'n' in o and not (isinstance(o['n'], int) and 1 <= o['n'] <= LIMITS['steps']): return 'a bad step number'
    if op == 'demo_set': return _demo_why(o, ctx)
    if op == 'video_at':
        v, t, q = ctx.get('video') or {}, o.get('t'), o.get('quote')
        if not v.get('cues'): return 'no video on the page'
        if not (_num(t) and t >= 0 and isinstance(q, str) and len(q) <= 200): return 'a bad video mark'
        if not _in_cues(v['cues'], t, q): return 'the quote is not in the transcript at that time'
    return ''

Good lines pass. The checks drop an unknown op, an unknown key, a link and markup:

In [ ]:
test_eq(check_line(dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t'), w=0), ctx), '')
test_eq(check_line(dict(op='label', at=dict(a='figure', label='a'), text='side a', side='left'), ctx), '')
test_eq(check_line(dict(op='arrow', **{'from': dict(a='m1', pt=[1100, 500]), 'to': dict(a='m1', tex=r'(1-\bar\alpha_t)')}), ctx), '')
test_eq(check_line(dict(op='clear'), ctx), '')
assert check_line(dict(op='script', code='alert(1)'), ctx)
assert check_line(dict(op='circle', at='passage', html='<b>'), ctx)
assert check_line(dict(op='label', at='passage', text='see https://evil.example'), ctx)
assert check_line(dict(op='label', at='passage', text='<img src=x onerror=alert(1)>'), ctx)
assert check_line(dict(op='step', say='x' * 161), ctx)

A sketch has at most 64 points:

In [ ]:
test_eq(check_line(dict(op='sketch', at=dict(a='m1', pt=[1050, 0]), pts=[[0, 0], [1000, 1000]]), ctx), '')
assert check_line(dict(op='sketch', at='m1', pts=[[i, i] for i in range(65)]), ctx)

The demo gate: `demo_set` drops while the learner has not locked in a prediction, so the board cannot give the result
first. After the lock-in it changes only inputs that the demo has, in their range.

In [ ]:
assert 'gate' in check_line(dict(op='demo_set', set=dict(t=500)), ctx)
open_ctx = by_id({**ctx, 'demo': {**ctx.demo, 'open': True}})
test_eq(check_line(dict(op='demo_set', set=dict(t=500)), open_ctx), '')
assert check_line(dict(op='demo_set', set=dict(beta=1)), open_ctx)
assert check_line(dict(op='demo_set', set=dict(t=5000)), open_ctx)

`video_at` quotes the transcript at its time, word for word:

In [ ]:
vctx = by_id({**ctx, 'video': dict(id='abc123def45', cues=[[0, 'Today we add noise.'], [61.5, 'Alpha bar is the share of the signal that is left.'], [70, 'It falls as t grows.']])})
test_eq(check_line(dict(op='video_at', t=63, quote='the share of the signal'), vctx), '')
assert check_line(dict(op='video_at', t=63, quote='the share of the noise'), vctx)
assert check_line(dict(op='video_at', t=63, quote='Alpha bar'), vctx)      # too short to be a quote
assert check_line(dict(op='video_at', t=63, quote='the share of the signal'), ctx)   # no video on the page

## The whole reply

`parse_board` reads a reply line by line. Each step keeps at most 6 strokes, and a board has at most 6 steps. A stroke
before the first step drops. Two dropped lines in a row end the board (`failed`): the steps before them stay.

A step can quote a source: `src` is the number of the source in `ctx.sources`, and `quote` must be word for word in its
text, with 40 to 600 characters (`research.verbatim`, the gate that research uses). A quote that fails drops, and the
step stays, in the tutor's words.

In [ ]:
#| export
def _quote(o, ctx):
    "The source and the word-for-word quote of step line `o`, or None."
    from abhyasa.research import verbatim
    src, q, ss = o.get('src'), o.get('quote'), ctx.get('sources') or []
    if not (isinstance(src, int) and 1 <= src <= len(ss) and isinstance(q, str)): return None
    if not (v := verbatim(q, ss[src - 1].get('text') or '')): return None
    return dict(src=src, text=v)

def parse_board(text:str, ctx) -> AttrDict:
    "The checked board in the model's reply `text`: `steps`, `dropped` lines with why, `sure`, `need` and `failed`."
    out = AttrDict(steps=[], dropped=[], sure=None, need='none', failed=False)
    run = 0
    lines = [l.strip() for l in str(text or '').splitlines() if l.strip() and not l.strip().startswith('```')]
    for k, line in enumerate(lines[:LIMITS['lines']]):
        why = ''
        try: o = json.loads(line) if len(line.encode()) <= LIMITS['line_bytes'] else None
        except ValueError: o = None
        if o is None: why = 'not one JSON object of 600 bytes or less'
        elif isinstance(o, dict) and o.get('op') == 'v' and k == 0:
            if o.get('n', VERSION) > VERSION: out.failed = True; out.dropped.append((k, 'a newer board version')); break
            if _num(o.get('sure')): out.sure = max(0., min(1., float(o['sure'])))
            if o.get('need') in NEEDS: out.need = o['need']
            continue
        elif not (why := check_line(o, ctx)):
            if o['op'] == 'step':
                if len(out.steps) >= LIMITS['steps']: why = 'more than 6 steps'
                else:
                    s = dict(n=len(out.steps) + 1, say=o['say'].strip(), strokes=[])
                    if (q := _quote(o, ctx)): s['quote'] = q
                    elif 'quote' in o: out.dropped.append((k, 'the quote is not word for word in its source'))
                    out.steps.append(s)
            elif not out.steps: why = 'a stroke before the first step'
            elif len(out.steps[-1]['strokes']) >= LIMITS['strokes']: why = 'more than 6 strokes in a step'
            else: out.steps[-1]['strokes'].append({k2: v for k2, v in o.items() if k2 != 'n' or o['op'] == 'clear'})
        if why:
            out.dropped.append((k, why)); run += 1
            if run >= 2: out.failed = True; break
        else: run = 0
    return out

A board for "I don't get alpha bar" on the DDPM formula: three steps, each with its strokes. The header says that the
tutor is sure.

In [ ]:
reply = '\n'.join(json.dumps(o) for o in [
    dict(op='v', n=1, sure=0.9, need='none'),
    dict(op='step', say='This formula adds noise in one jump.'), dict(op='box', at='m1', w=2),
    dict(op='step', say='Alpha bar t is how much of the clean picture is left at step t.'),
    dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t'), w=0), dict(op='label', at=dict(a='m1', tex=r'\bar\alpha_t'), text='how much is left', side='above', w=5),
    dict(op='step', say='It starts near one and falls to near zero.'), dict(op='sketch', at=dict(a='m1', pt=[1050, 0]), pts=[[0, 100], [500, 500], [1000, 950]])])
b = parse_board(reply, ctx)
test_eq(len(b.steps), 3)
test_eq([len(s['strokes']) for s in b.steps], [1, 2, 1])
test_eq((b.sure, b.need, b.failed, b.dropped), (0.9, 'none', False, []))

A bad line drops alone, and the next line still draws. Two bad lines in a row end the board:

In [ ]:
b = parse_board('{"op":"step","say":"One."}\n{"op":"circle","at":"e7"}\n{"op":"circle","at":"m1"}', ctx)
test_eq((len(b.steps[0]['strokes']), len(b.dropped), b.failed), (1, 1, False))
b = parse_board('{"op":"step","say":"One."}\nnot json\n{"op":"circle","at":"e7"}\n{"op":"step","say":"Two."}', ctx)
test_eq((len(b.steps), b.failed), (1, True))

At most 6 strokes in a step and 6 steps in a board; a stroke before the first step drops:

In [ ]:
many = '\n'.join(['{"op":"step","say":"One."}'] + ['{"op":"circle","at":"m1"}'] * 6 + ['{"op":"box","at":"m1"}', '{"op":"step","say":"Two."}'])
b = parse_board(many, ctx)
test_eq(([len(s['strokes']) for s in b.steps], len(b.dropped)), ([6, 0], 1))
test_eq(len(parse_board('\n'.join(['{"op":"step","say":"A step."}'] * 7), ctx).steps), 6)
test_eq(parse_board('{"op":"circle","at":"m1"}\n{"op":"step","say":"One."}', ctx).dropped[0][1], 'a stroke before the first step')

A quote stays only when it is word for word in its source:

In [ ]:
qctx = by_id({**ctx, 'sources': [dict(title='DDPM', text='A notable property of the forward process is that it admits sampling x_t at an arbitrary timestep t in closed form.')]})
good = dict(op='step', say='The paper says it.', src=1, quote='it admits sampling x_t at an arbitrary timestep t in closed form')
test_eq(parse_board(json.dumps(good), qctx).steps[0]['quote']['src'], 1)
b = parse_board(json.dumps({**good, 'quote': 'it allows sampling x_t at any timestep t in closed form, fast'}), qctx)
assert 'quote' not in b.steps[0] and b.dropped

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()